# BME 574 — Convolutional neural networks
## Part 1: it works.  Part 2: why.

We are going to train a network to read handwritten digits, and it is going to be very good at it
almost immediately. Then we are going to take the thing apart and find out what it is doing, and
end on an experiment that shows exactly what a convolution buys you — and what it costs.

**Runtime.** About six minutes end to end on a laptop. Three cells do almost all of it: the first
training run, and the two comparison runs in §6 and §7.

**The dataset.** MNIST: 70,000 handwritten digits, 28×28 greyscale, collected by NIST from Census
Bureau employees and American high-school students and assembled into this form by LeCun, Cortes
and Burges. It is the most-used dataset in the history of machine learning, it is the one the first
convolutional networks were built for, and it is small enough to train on a laptop in a minute.
`torchvision` ships a loader for it, which is what we use below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset, DataLoader

plt.rcParams.update({'font.size': 9, 'figure.dpi': 120})
torch.manual_seed(0)
SEED, EPOCHS, BATCH, LR = 0, 6, 128, 1e-3

### Loading it

The standard `torchvision` call. The first run downloads about 11 MB into `./data` and every run
after that uses the copy on disk.

*If you are on a machine with no network* — a locked-down lab image, say — put the four original
files (`train-images-idx3-ubyte.gz`, `train-labels-idx1-ubyte.gz`, `t10k-images-idx3-ubyte.gz`,
`t10k-labels-idx1-ubyte.gz`) into `./data/MNIST/raw/` and the same call will find them and skip the
download. Nothing else changes.

In [ ]:
# MNIST's canonical per-pixel mean and standard deviation. You will see these
# two numbers in every MNIST example ever written; we check them against the
# data further down rather than taking them on faith.
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

transform = transforms.Compose([
    transforms.ToTensor(),                                 # uint8 HxW -> float32 1xHxW in [0,1]
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,)),     # -> roughly zero mean, unit sd
])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print('train', len(train_dataset), ' test', len(test_dataset))
img, label = train_dataset[0]
print('one sample:', tuple(img.shape), img.dtype, ' label', label)
print('after Normalize, pixel range %.3f .. %.3f' % (img.min(), img.max()))
print('   (0 and 1 map to (0-%.4f)/%.4f = %.3f and (1-%.4f)/%.4f = %.3f)'
      % (MNIST_MEAN, MNIST_STD, (0 - MNIST_MEAN) / MNIST_STD,
         MNIST_MEAN, MNIST_STD, (1 - MNIST_MEAN) / MNIST_STD))

### One detour, because the rest of the notebook needs it

`train_loader` above is the normal way to feed a network, and it is what you will use in the
project. This notebook is unusual: it trains the **same data six different ways** — permuted,
shifted, with two different architectures — and re-scores the whole test set each time. For that it
is much easier to hold the images as one big tensor than to keep rebuilding `Dataset` objects, and
it also runs about twice as fast, because `ToTensor` and `Normalize` otherwise run in Python once
per image per epoch.

`train_dataset.data` is the raw `uint8` tensor straight from the file, and `.targets` the labels,
so we can apply the same two steps once, to everything. The cell asserts that the result is
**identical** to what the transform pipeline produces — worth checking rather than assuming.

In [ ]:
Xtr_u8, ytr_t = train_dataset.data, train_dataset.targets      # (60000,28,28) uint8
Xte_u8, yte_t = test_dataset.data, test_dataset.targets

def to_tensor(u8):
    return ((u8.float() / 255.0) - MNIST_MEAN) / MNIST_STD     # same two steps as `transform`

Xtr = to_tensor(Xtr_u8).unsqueeze(1)        # (N, channels, height, width) -- what Conv2d wants
Xte = to_tensor(Xte_u8).unsqueeze(1)
ytr, yte = ytr_t.clone(), yte_t.clone()

# prove it is the same thing the DataLoader would have handed us
for i in (0, 1, 12345):
    assert torch.equal(train_dataset[i][0], Xtr[i]), 'mismatch at %d' % i
    assert train_dataset[i][1] == int(ytr[i])
print('verified: Xtr[i] is bit-identical to train_dataset[i] for i = 0, 1, 12345')

# and check the two famous constants against the data itself
raw_mean = (Xtr_u8.float() / 255.0).mean()
raw_std = (Xtr_u8.float() / 255.0).std()
print('measured training mean %.4f, sd %.4f   (the constants above: %.4f, %.4f)'
      % (raw_mean, raw_std, MNIST_MEAN, MNIST_STD))
print()
print('tensor shape %s  |  labels %s, counts %s'
      % (tuple(Xtr.shape), tuple(ytr.shape), np.bincount(ytr.numpy())))

# numpy views for the plotting cells
ytr_np, yte_np = ytr.numpy(), yte.numpy()
Xtr_u8_np, Xte_u8_np = Xtr_u8.numpy(), Xte_u8.numpy()

In [ ]:
fig, ax = plt.subplots(2, 10, figsize=(10, 2.3))
for c in range(10):
    hits = np.where(ytr_np == c)[0][:2]
    for r in range(2):
        ax[r, c].imshow(Xtr_u8_np[hits[r]], cmap='gray_r')
        ax[r, c].set_xticks([]); ax[r, c].set_yticks([])
    ax[0, c].set_title(str(c), fontsize=10)
fig.suptitle('two examples of each digit', y=1.02, fontsize=10)
plt.tight_layout(); plt.show()

---
# 1 · It works

Here is the whole model. Read it top to bottom as a pipeline: an image goes in at the left, and ten
numbers come out at the right.

```
(1, 28, 28)  ->  Conv2d(1, 16, 3) -> ReLU -> MaxPool(2)  ->  (16, 14, 14)
             ->  Conv2d(16, 32, 3) -> ReLU -> MaxPool(2) ->  (32,  7,  7)
             ->  Flatten -> Linear(1568, 10)             ->  (10,) logits
```

Do not worry yet about what `Conv2d` does — that is §3. For now just note that it is **twenty
thousand parameters**, which is small, and that the last layer is the same `Linear` layer you used
in Project 3, taking a flattened vector and producing one logit per class.

In [ ]:
def make_cnn(seed=SEED):
    torch.manual_seed(seed)
    return nn.Sequential(
        nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 28x28 -> 14x14
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14x14 ->  7x7
        nn.Flatten(),
        nn.Linear(32 * 7 * 7, 10))


def n_params(m):
    return sum(p.numel() for p in m.parameters())


def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        pred = torch.cat([model(X[i:i + 2000]).argmax(1) for i in range(0, len(X), 2000)])
    return float((pred == y).double().mean()), pred


def train(model, Xa, ya, Xv, yv, epochs=EPOCHS, seed=SEED, verbose=True):
    loader = DataLoader(TensorDataset(Xa, ya), batch_size=BATCH, shuffle=True,
                        generator=torch.Generator().manual_seed(seed + 100))
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    lossfn = nn.CrossEntropyLoss()
    hist = []
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            lossfn(model(xb), yb).backward()
            opt.step()
        a, _ = accuracy(model, Xv, yv)
        hist.append(a)
        if verbose:
            print('   epoch %d   test accuracy %.4f' % (ep + 1, a))
    return np.array(hist)


cnn = make_cnn()
print('%s parameters\n' % format(n_params(cnn), ','))
import time; t0 = time.time()
hist_cnn = train(cnn, Xtr, ytr, Xte, yte)
acc_cnn, pred_cnn = accuracy(cnn, Xte, yte)
print('\nfinal test accuracy %.4f   (%d wrong out of %d)   trained in %.0f s'
      % (acc_cnn, int((pred_cnn != yte).sum()), len(yte), time.time() - t0))

### Look at what it got wrong

Roughly one in a hundred. Before reading on, look at these and ask how many of them *you* would
have got right with no context.

In [ ]:
wrong = np.where((pred_cnn != yte).numpy())[0]
show = wrong[:30]
fig, ax = plt.subplots(3, 10, figsize=(10, 3.4))
for k, a in enumerate(ax.ravel()):
    a.set_xticks([]); a.set_yticks([])
    if k >= len(show):
        a.axis('off'); continue
    i = show[k]
    a.imshow(Xte_u8_np[i], cmap='gray_r')
    a.set_title('%d, said %d' % (yte_np[i], pred_cnn[i]), fontsize=7.5, color='#9C2B2B')
fig.suptitle('the first %d of %d mistakes' % (len(show), len(wrong)), y=1.01, fontsize=10)
plt.tight_layout(); plt.show()

That took about a minute and twenty lines of model definition. **Now the interesting part: what
did `Conv2d` actually do?**

---
# 2 · What one convolution does

A convolution slides a small grid of weights — a **kernel** — across the image. At every position
it multiplies the kernel elementwise with the patch underneath and sums. One kernel produces one
output image, called a **feature map**.

Two things follow immediately, and they are the whole idea:

1. **It is local.** A 3×3 kernel only ever looks at 9 neighbouring pixels at a time. It is built
   into the operation that nearby pixels belong together — which is true of images and false of,
   say, the 21 columns of the thyroid panel in Project 3.
2. **The weights are shared.** The *same* nine numbers are applied at all 784 positions. A feature
   detector learned in the top-left corner works in the bottom-right corner for free.

Here are four kernels chosen by hand, so you can see what a feature map is before we let the
network choose its own.

In [ ]:
digit = (Xte_u8[0].float() / 255.0)[None, None]

KERNELS = {
    'vertical edge':   [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],      # Sobel
    'horizontal edge': [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
    'blur':            [[1 / 9] * 3] * 3,
    'sharpen':         [[0, -1, 0], [-1, 5, -1], [0, -1, 0]],
}

fig, ax = plt.subplots(2, 5, figsize=(11, 5.0))
ax[0, 0].imshow(digit[0, 0], cmap='gray_r'); ax[0, 0].set_title('input', fontsize=9)
ax[1, 0].axis('off')
for j, (name, k) in enumerate(KERNELS.items(), start=1):
    w = torch.tensor(k, dtype=torch.float32)[None, None]
    out = F.conv2d(digit, w, padding=1)[0, 0]
    ax[0, j].imshow(out, cmap='gray_r')
    ax[0, j].set_title(name, fontsize=9)
    ax[1, j].imshow(w[0, 0], cmap='RdBu_r', vmin=-2.2, vmax=2.2)
    ax[1, j].set_title('the 3x3 kernel', fontsize=8)
    for i in range(3):
        for jj in range(3):
            ax[1, j].text(jj, i, '%.2g' % w[0, 0, i, jj], ha='center', va='center', fontsize=7)
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
fig.suptitle('one image, four kernels: each produces one feature map', y=1.0, fontsize=10)
plt.tight_layout(h_pad=1.8); plt.show()

print('The vertical-edge kernel fires on left-right brightness changes and ignores')
print('flat regions. Nine numbers. Applied at all 784 positions. That is a convolution.')

### The parameter arithmetic, which is the real argument

Suppose you wanted a dense layer to transform a 28×28 image into another 28×28 image. Every output
pixel connects to every input pixel:

| | weights |
|---|---|
| `Linear(784, 784)` | 784 × 784 + 784 = **615,440** |
| one 3×3 convolution kernel | 3 × 3 + 1 = **10** |

Sixty-one thousand times fewer. And the convolution is not a cheap approximation of the dense
layer — it is a *different hypothesis*: that the useful transformation is the same everywhere in
the image. When that hypothesis is right, you have thrown away 615,430 parameters you did not need.

The cell below counts it for the model we actually trained.

In [ ]:
print('%-34s %10s   %s' % ('layer', 'params', 'why'))
print('-' * 78)
rows = [
    ('Conv2d(1, 16, 3)',   1 * 16 * 9 + 16,      '16 kernels x (1 channel x 9) + 16 biases'),
    ('Conv2d(16, 32, 3)',  16 * 32 * 9 + 32,     '32 kernels x (16 channels x 9) + 32 biases'),
    ('Linear(1568, 10)',   1568 * 10 + 10,       'the classifier head, and 77% of the model'),
]
for name, p, why in rows:
    print('%-34s %10s   %s' % (name, format(p, ','), why))
print('-' * 78)
print('%-34s %10s' % ('total', format(sum(r[1] for r in rows), ',')))
print()
print('A Linear layer doing the FIRST convolution\'s job, position by position:')
print('   Linear(784, 784) = %s weights, versus %d for the kernel.'
      % (format(784 * 784 + 784, ','), 1 * 1 * 9 + 1))

### What the network actually learned

Those were kernels we chose. Here are the sixteen the network found for itself, and what the first
layer does to one digit. Nobody told it to look for strokes and edges.

In [ ]:
W = cnn[0].weight.detach()           # (16, 1, 3, 3)
lim = float(W.abs().max())
fig, ax = plt.subplots(2, 8, figsize=(9, 2.6))
for i, a in enumerate(ax.ravel()):
    a.imshow(W[i, 0], cmap='RdBu_r', vmin=-lim, vmax=lim)
    a.set_xticks([]); a.set_yticks([])
fig.suptitle('the 16 learned first-layer kernels (red positive, blue negative)',
             y=1.04, fontsize=10)
plt.tight_layout(); plt.show()

with torch.no_grad():
    maps = F.relu(cnn[0](digit))[0]   # (16, 28, 28)
fig, ax = plt.subplots(2, 9, figsize=(10.5, 3.4))
ax[0, 0].imshow(digit[0, 0], cmap='gray_r'); ax[0, 0].set_title('input', fontsize=8)
ax[1, 0].axis('off')
for i in range(16):
    a = ax[(i // 8), 1 + i % 8]
    a.imshow(maps[i], cmap='viridis')
    a.set_title('map %d' % i, fontsize=7)
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
fig.suptitle('what the first layer sees: 16 feature maps of one digit', y=1.02, fontsize=10)
plt.tight_layout(h_pad=1.6); plt.show()

---
# 3 · Pooling, and how much of the image a unit can see

`MaxPool2d(2)` takes each 2×2 block and keeps the largest value. It does two things: it halves the
width and height (so the next layer is four times cheaper), and it makes the output slightly
insensitive to *where* in that 2×2 block the feature was.

The second effect is the one people quote, and it is weaker than it sounds. **Two max-pools buy
you about two pixels of tolerance, not translation invariance.** We will measure exactly how much
in §8.

### Receptive field

A unit deep in the network sees only what fed into it. Trace it back:

In [ ]:
print('%-26s %8s %10s %s' % ('after', 'size', 'rec.field', 'what one unit there can see'))
print('-' * 82)
rows = [('input',          28, 1,  'one pixel'),
        ('Conv2d 3x3',     28, 3,  'a 3x3 patch'),
        ('MaxPool 2',      14, 4,  'a 4x4 patch'),
        ('Conv2d 3x3',     14, 8,  'an 8x8 patch'),
        ('MaxPool 2',       7, 10, 'a 10x10 patch -- about a third of the image')]
for n, s, r, w in rows:
    print('%-26s %5dx%-2d %7dx%-3d %s' % (n, s, s, r, r, w))
print()
print('So no single unit before the Linear layer has seen a whole digit. The classifier')
print('head is what combines 7x7x32 local verdicts into one answer. Stack more')
print('conv-pool blocks and the receptive field grows until it covers the image --')
print('which is most of what "deeper" buys you.')

with torch.no_grad():
    before = F.relu(cnn[0](digit))[0, :4]
    after = cnn[2](F.relu(cnn[0](digit)))[0, :4]
fig, ax = plt.subplots(2, 4, figsize=(7.5, 4))
for i in range(4):
    ax[0, i].imshow(before[i], cmap='viridis'); ax[0, i].set_title('28x28', fontsize=8)
    ax[1, i].imshow(after[i], cmap='viridis'); ax[1, i].set_title('14x14 after pool', fontsize=8)
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
fig.suptitle('max-pooling: same features, half the resolution', y=1.0, fontsize=10)
plt.tight_layout(); plt.show()

---
# 4 · Is the convolution actually doing the work?

So far this is a story. Here is the test.

We will train two dense networks on exactly the same data with exactly the same optimiser:

- one with **the same number of parameters** as the CNN (784 → 26 → 10, about 20,700);
- one with **eleven times more** (784 → 256 → 128 → 10, about 235,000).

If the CNN's advantage were really about capacity, the big dense net should close the gap.

*This cell trains two models; it takes about 15 seconds.*

In [ ]:
def make_dense(widths, seed=SEED):
    torch.manual_seed(seed)
    sizes = [784] + list(widths)
    layers = [nn.Flatten()]
    for a, b in zip(sizes[:-1], sizes[1:]):
        layers += [nn.Linear(a, b), nn.ReLU()]
    layers.append(nn.Linear(sizes[-1], 10))
    return nn.Sequential(*layers)


results = {}
for name, make in (('CNN (from §1)', None),
                   ('dense 784-26-10', lambda: make_dense([26])),
                   ('dense 784-256-128-10', lambda: make_dense([256, 128]))):
    if make is None:
        results[name] = (acc_cnn, n_params(cnn))
        continue
    m = make()
    train(m, Xtr, ytr, Xte, yte, verbose=False)
    results[name] = (accuracy(m, Xte, yte)[0], n_params(m))

print('%-24s %10s %10s' % ('model', 'params', 'test acc'))
print('-' * 48)
for k, (a, p) in results.items():
    print('%-24s %10s %10.4f' % (k, format(p, ','), a))
print()
print('CNN minus matched-parameter dense : %+.4f'
      % (results['CNN (from §1)'][0] - results['dense 784-26-10'][0]))
print('CNN minus %.0fx-larger dense       : %+.4f'
      % (results['dense 784-256-128-10'][1] / results['CNN (from §1)'][1],
         results['CNN (from §1)'][0] - results['dense 784-256-128-10'][0]))

---
# 5 · The experiment that settles it

Here is the cleanest demonstration of what a convolution actually assumes.

Take **one** random permutation of the 784 pixel positions. Apply that *same* permutation to every
image in the training set and every image in the test set. No information has been destroyed — the
mapping is a fixed bijection, and a lookup table could undo it perfectly. The images are now
unreadable to you, but every pixel value is still there, and every image is scrambled identically.

**A dense network should not care at all.** Its first layer is a full matrix: permuting the inputs
just permutes its columns, and it can learn the permuted columns exactly as easily as the original
ones.

**A CNN should fall apart.** Everything it assumes — that neighbouring pixels are related, that a
feature detector is worth reusing across positions — is now false. Its 3×3 window covers nine
pixels that came from nine unrelated places.

Predict the two numbers before you run it.

In [ ]:
g = torch.Generator().manual_seed(0)
PERM = torch.randperm(784, generator=g)

def permute_pixels(X, perm=PERM):
    return X.reshape(len(X), -1)[:, perm].reshape(X.shape)

pXtr, pXte = permute_pixels(Xtr), permute_pixels(Xte)

fig, ax = plt.subplots(2, 8, figsize=(9, 2.6))
for i in range(8):
    ax[0, i].imshow(Xte[i, 0], cmap='gray_r')
    ax[1, i].imshow(pXte[i, 0], cmap='gray_r')
    ax[0, i].set_title(str(yte_np[i]), fontsize=9)
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
ax[0, 0].set_ylabel('original', fontsize=8); ax[1, 0].set_ylabel('permuted', fontsize=8)
fig.suptitle('same pixels, same labels, one fixed shuffle of the 784 positions',
             y=1.03, fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# retrain both architectures from scratch on the permuted data
# -- this is the slow cell, about 70 seconds
perm_res = {}
for name, make in (('CNN (16-32 conv)', make_cnn),
                   ('dense 784-256-128-10', lambda: make_dense([256, 128]))):
    m = make()
    train(m, pXtr, ytr, pXte, yte, verbose=False)
    perm_res[name] = accuracy(m, pXte, yte)[0]

orig = {'CNN (16-32 conv)': results['CNN (from §1)'][0],
        'dense 784-256-128-10': results['dense 784-256-128-10'][0]}

print('%-24s %12s %12s %10s' % ('model', 'original', 'permuted', 'change'))
print('-' * 62)
for k in orig:
    print('%-24s %12.4f %12.4f %+10.4f' % (k, orig[k], perm_res[k], perm_res[k] - orig[k]))
print()
print('And the comparison that matters, on the permuted data:')
print('   CNN %.4f   vs   dense %.4f   ->  %+.4f'
      % (perm_res['CNN (16-32 conv)'], perm_res['dense 784-256-128-10'],
         perm_res['CNN (16-32 conv)'] - perm_res['dense 784-256-128-10']))

### What that means

The dense net did not notice. The CNN lost several points — and on the scrambled images it is now
**worse** than the dense network it comfortably beat on the real ones.

So the convolution was never a generically better layer. It is a **constraint**: the architecture
hard-codes an assumption that neighbouring inputs are related and that a useful feature is worth
detecting everywhere. On real images that assumption is true, and a model that assumes it needs far
fewer parameters and far less data to find the same structure. On scrambled images the assumption
is false, and the same constraint is now a handicap — the CNN is forbidden from using relationships
the dense net can learn freely.

> **This is the idea Project 4 is built on.** Choosing an architecture is choosing an assumption
> about your data. The question is never "is a CNN better" — it is "is the assumption a CNN makes
> true of *these* images?"

Hold on to that, because for medical images the answer is sometimes subtle.

---
# 6 · How much translation invariance did we actually get?

People say CNNs are translation invariant. Let us check, because it is only partly true and the
part that is false is the part that matters clinically.

We take the models we already trained — **no retraining, no augmentation** — and shift every test
image by a few pixels.

In [ ]:
def shift(X, d):
    # translate down-right by d pixels, filling the vacated edge with the
    # background value -- NOT torch.roll, which would wrap the image around
    if d == 0:
        return X
    out = torch.full_like(X, (0.0 - MNIST_MEAN) / MNIST_STD)
    out[:, :, d:, d:] = X[:, :, :-d, :-d]
    return out

dense_big = make_dense([256, 128]); train(dense_big, Xtr, ytr, Xte, yte, verbose=False)

shifts = [0, 1, 2, 3, 4]
rows = {'CNN': [accuracy(cnn, shift(Xte, d), yte)[0] for d in shifts],
        'dense 784-256-128-10': [accuracy(dense_big, shift(Xte, d), yte)[0] for d in shifts]}

print('%-24s %s' % ('shift (px)', ''.join('%8d' % d for d in shifts)))
print('-' * 66)
for k, v in rows.items():
    print('%-24s %s' % (k, ''.join('%8.3f' % a for a in v)))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.1))
for (k, v), col in zip(rows.items(), ['#1F3864', '#9C2B2B']):
    ax[0].plot(shifts, v, 'o-', color=col, label=k)
ax[0].set_xlabel('diagonal shift, pixels'); ax[0].set_ylabel('test accuracy')
ax[0].legend(fontsize=7.5); ax[0].grid(alpha=.3); ax[0].set_ylim(0, 1)
ax[0].set_title('neither model is translation invariant', fontsize=9)
ax[1].imshow(np.hstack([shift(Xte[:1], d)[0, 0].numpy() for d in (0, 2, 4)]), cmap='gray_r')
ax[1].set_xticks([]); ax[1].set_yticks([])
ax[1].set_title('the same digit shifted 0, 2 and 4 px', fontsize=9)
plt.tight_layout(); plt.show()

Both collapse. The CNN holds up better at every shift — the pooling does buy something — but by
four pixels, one seventh of the image width, both are useless.

The lesson is not that pooling fails. It is that **an invariance you want has to come from
somewhere**, and there are only two places to get it:

1. **Build it into the architecture** — pooling, or a genuinely equivariant design.
2. **Put it in the data** — show the network shifted, rotated, flipped copies during training.
   That is data augmentation, and it is usually the cheaper and stronger of the two.

And now the question that makes this a medical imaging problem rather than a digits problem:
**which transformations leave the label unchanged for *your* modality?**

- A blood cell on a slide has no up. Rotate it, flip it, mirror it — it is the same cell.
- A skin lesion has no up either, but its **colour** is diagnostic, so a careless brightness or
  hue jitter can destroy the signal you need.
- A CT slice has a definite left and right. Mirror it and the liver moves to the wrong side —
  and if two of your classes are *femur-left* and *femur-right*, a horizontal flip has just
  relabelled your data.

Getting that list right for one modality is the work of Project 4.

---
## Summary

| | |
|---|---|
| **A convolution** | a small kernel slid over the image: local, and the same weights everywhere |
| **Why it is cheap** | 10 parameters where a dense layer would need 615,440 |
| **Why it is good** | because the assumption behind it — nearby pixels are related, features recur — is true of images |
| **Pooling** | halves the resolution, buys about two pixels of position tolerance |
| **Receptive field** | how much of the input one deep unit can see; grows with depth |
| **The real lesson** | the architecture *is* an assumption. Scramble the pixels and the CNN loses to a dense net. |
| **Invariance** | comes from the architecture or from augmentation, and which ones you want depends on the modality |

### What we did not cover

Batch normalisation, residual connections, dilated and strided convolutions, 1×1 convolutions,
and transfer learning from a pretrained backbone. The project does not need them; several would
be a good final-project direction.

### Next

`BME574_Project4_explained.ipynb`, then the Project 4 starter. Same CNN, real medical images, and
a modality where you have to decide for yourself which invariances are true.